In [ ]:
import os
import random
import re
import shutil
import urllib.request
from collections import defaultdict, Counter

In [ ]:
CORPUS_URL = "https://www.gutenberg.org/files/11/11-0.txt"
req = urllib.request.Request(CORPUS_URL, headers={"User-Agent": "Mozilla/5.0"})
raw = urllib.request.urlopen(req, timeout=60).read().decode("utf-8")
start = raw.find("*** START")
start = raw.find("\n", start) + 1
end = raw.find("*** END")
text = re.sub(r"\s+", " ", raw[start:end]).strip()
print(len(text), "characters,", len(text.split()), "words")
print(text[:300])

In [ ]:
class MarkovChain:
    def __init__(self, order=2, level="word"):
        self.order = order
        self.level = level
        self.transitions = defaultdict(Counter)
        self.starts = []

    def _tokenize(self, text):
        if self.level == "word":
            return text.split()
        return list(text)

    def _join(self, tokens):
        if self.level == "word":
            return " ".join(tokens)
        return "".join(tokens)

    def train(self, text):
        tokens = self._tokenize(text)
        for i in range(len(tokens) - self.order):
            state = tuple(tokens[i:i + self.order])
            nxt = tokens[i + self.order]
            self.transitions[state][nxt] += 1
            if tokens[i][:1].isupper():
                self.starts.append(state)
        if not self.starts:
            self.starts = list(self.transitions.keys())

    def generate(self, length=100, seed_state=None):
        state = seed_state if seed_state else random.choice(self.starts)
        output = list(state)
        while len(output) < length:
            counter = self.transitions.get(state)
            if not counter:
                state = random.choice(self.starts)
                output.extend(state)
                continue
            candidates = list(counter.keys())
            weights = list(counter.values())
            nxt = random.choices(candidates, weights=weights, k=1)[0]
            output.append(nxt)
            state = tuple(output[-self.order:])
        return self._join(output[:length])

In [ ]:
random.seed(42)
shutil.rmtree("outputs", ignore_errors=True)
os.makedirs("outputs")

In [ ]:
word_model = MarkovChain(order=2, level="word")
word_model.train(text)
example_state = ("Alice", "was")
print(example_state, "->", dict(word_model.transitions[example_state]))

In [ ]:
word_lines = []
for order in (1, 2, 3):
    model = MarkovChain(order=order, level="word")
    model.train(text)
    sample = model.generate(length=60)
    word_lines.append(f"[word-level | order={order}]\n{sample}\n")
    print(word_lines[-1])
with open("outputs/word_level_samples.txt", "w") as f:
    f.write("\n".join(word_lines))

In [ ]:
char_lines = []
for order in (2, 4, 8):
    model = MarkovChain(order=order, level="char")
    model.train(text)
    sample = model.generate(length=300)
    char_lines.append(f"[char-level | order={order}]\n{sample}\n")
    print(char_lines[-1])
with open("outputs/char_level_samples.txt", "w") as f:
    f.write("\n".join(char_lines))

In [ ]:
seeded = MarkovChain(order=2, level="word")
seeded.train(text)
print(seeded.generate(length=60, seed_state=("Alice", "was")))
print()
print(seeded.generate(length=60, seed_state=("The", "Queen")))

In [ ]:
archive_path = shutil.make_archive("markov_outputs", "zip", "outputs")
print("created", archive_path)
try:
    from google.colab import files
    files.download("markov_outputs.zip")
except ImportError:
    pass